# Why the pasted-Abacus galaxy auto-power $C_\ell^{gg}$ exceeds GODMAX theory

**Stage-31, pz3 cap2400, nside2048, lmax3000, 13-log bandpowers — checkpoint_000550 bestfit**

## Executive summary

The pasted-Abacus galaxy clustering power $C_\ell^{gg}$ sits **~9% above theory at large scales and rises to ~64% by $\ell\simeq900\text{--}1500$**, even though the paste and the theory use the **same** HOD parameters, the **same** profile machinery, and the **same** params file. This notebook shows the excess is **not** a painting bug, **not** a measurement error, and **not** an $n(z)$ effect. It decomposes cleanly into two physical pieces:

| Component | Effect on $C_\ell^{gg}$ | In bias terms | Verdict |
|---|---|---|---|
| **2-halo (large scale, $\ell$-flat)** | **+16%** (amplitude ratio $a_{2h}=1.16$) | $b_{\rm sim}/b_{\rm th}=1.076$ (**+7.6%**) | Abacus halos slightly more biased than analytic Tinker-2010 — *the "expected 10-15% from HMF"* |
| **1-halo (small scale, dominant at high $\ell$)** | **×1.76** (amplitude ratio $a_{1h}=1.76$) | — | **The main effect:** $n_{\rm gal}$ normalization mismatch amplified by $1/\bar n^2$ |

### The resolution of the puzzle

The HOD mass threshold $M_*^{\rm thresh}(z)$ is solved so the integral over the **analytic Tinker HMF** reproduces the DESI **target** density (theory $\bar n = 344/\mathrm{deg}^2 \approx$ target $335$). But applying the *identical* HOD to the *actual Abacus halos* paints only **$253.8/\mathrm{deg}^2$ — a 24% deficit** (Abacus has fewer halos in the HOD-relevant mass range than Tinker predicts at this cosmology / halo definition).

The measured galaxy overdensity $\delta_g = n/\bar n_{\rm sim}-1$ self-normalizes by the **realized (low)** $\bar n_{\rm sim}$, while the theory 1-halo term divides by the **target** $\bar n_{\rm th}^2$. Because the 1-halo power scales as $1/\bar n^2$, the 24% deficit is amplified **quadratically**:
$$\frac{C_{1h}^{\rm sim}}{C_{1h}^{\rm th}} \simeq \left(\frac{\bar n_{\rm th}}{\bar n_{\rm sim}}\right)^2 = \left(\frac{1}{0.743}\right)^2 = 1.81,$$
matching the fitted 1-halo amplitude ratio $a_{1h}=1.76$ to 3%.

**A modest ~24% halo-abundance difference becomes a ~50-80% $C_\ell^{gg}$ excess** because the 1-halo term depends on $1/\bar n^2$ — which is exactly why it looks far larger than the naive "10-15% from the HMF." The user's intuition is correct: the *bias/HMF* part really is only ~7.6%; the rest is the quadratic $\bar n$ amplification of the 1-halo term.

### What is NOT the cause (ruled out below)
- **Profile / painting:** satellites are inverse-CDF sampled from the *identical* truncated-NFW $\rho_{\rm clm}$ (Diemer15 $c$–$M$) that the theory Fourier-transforms into $u(k|M)$. The 1-halo $C_\ell$ **shape** matches theory exactly (a single amplitude fits all high-$\ell$ bands) → the profile is right; only the **normalization** is off.
- **Shot noise:** correctly subtracted using the sim's own painted $\bar n$ (`cl_convention=shot_noise_subtracted_signal`).
- **Pixel window:** $W_{\rm pix}^2$ only drops to 0.86 at the very last band ($\ell=2640$) for nside2048 — negligible.
- **$n(z)$ projection:** $\int (dN/dz)^2 dz$ differs by only **1.4%** between the painted catalog and the theory kernel.


## 0. Setup and data products

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")          # CPU theory build
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("OMP_NUM_THREADS", "8")
import sys, json
from pathlib import Path
import numpy as np, h5py
import matplotlib.pyplot as plt

REPO = "/mnt/ceph/users/spandey/ltu-godmax/GODMAX"
RR   = f"{REPO}/data/xDESI/processed/abacus_backlight/stage31_pz3_cap2400_hmcbestfit_mmin11p147538_lmax3000_13log"
SIM  = f"{RR}/measurements/sim_pz3_cap2400_hmcbestfit_mmin11p147538_nside2048_lmax3000_nbin13_log.h5"
SUM  = f"{RR}/theory/stage31_pz3_cap2400_hmcbestfit_mmin11p147538_lmax3000_13log_theory_poweradd_sum_for_sim_measurement_matched_transfers.h5"
DATA = f"{REPO}/data/xDESI/processed/multiprobe_namaster_true_nz/midres2048/xdesi_multiprobe_cls_cov_nside2048_ell128_lmax3000_nbin13_log_apo1deg_C2_pairmean.h5"
NZH5 = f"{REPO}/data/xDESI/processed/multiprobe_namaster_true_nz/midres2048/xdesi_multiprobe_maps_nside2048_ell128_lmax3000_nbin13_log_apo1deg_C2_pairmean.h5"
MAP  = f"{REPO}/data/xDESI/processed/abacus_backlight/stage31_pz3_cap2400_lmax3000_gk1000_60param_warm100_checkpoint_000550/maps/stage31_pz3_cap2400_hmcbestfit_mmin11p147538_nside2048_lmax4096/abacus_pasted_maps_pz3cap2400_hmcbestfit_z0p63_0p98_logMgt11p147538_nside2048.h5"
PLOTS = Path(RR)/"plots"; PLOTS.mkdir(parents=True, exist_ok=True)
CONFIG = f"{REPO}/notebooks/xDESI/abacus_paste/stage31_pz3_cap2400_hmcbestfit_mmin11p147538_lmax3000_13log.selected.yaml"

# ---- load the measured sim galaxy auto + sim-matched theory (resolved poweradd) ----
with h5py.File(SIM,"r") as h5:
    ell = np.asarray(h5["ell"][:])
    sim_gg = np.asarray(h5["spectra/desi_g_auto_pz3/cl"][:])
    gg_attrs = dict(h5["spectra/desi_g_auto_pz3"].attrs)
with h5py.File(SUM,"r") as h5:
    nm = [s.decode() if isinstance(s,bytes) else s for s in h5["windowed/spectrum_names"][:]]
    nb = len(h5["windowed/ell"][:])
    th_gg = np.asarray(h5["windowed/resolved_log10Mgt11"][:]).reshape(len(nm),nb)[nm.index("desi_g_auto_pz3")]
obs = sim_gg/th_gg
print("galaxy-auto cl_convention :", gg_attrs.get("cl_convention"))
print("ell band centers          :", ell)
print("sim/theory (resolved, poweradd):")
for l,r in zip(ell, obs): print(f"   ell={l:7.1f}   sim/theory={r:.3f}")


galaxy-auto cl_convention : shot_noise_subtracted_signal
ell band centers          : [ 143.5  179.5  227.   287.   359.5  449.5  564.5  712.   897.  1157.
 1522.  2004.5 2640. ]
sim/theory (resolved, poweradd):
   ell=  143.5   sim/theory=1.092
   ell=  179.5   sim/theory=1.114
   ell=  227.0   sim/theory=1.268
   ell=  287.0   sim/theory=1.195
   ell=  359.5   sim/theory=1.327
   ell=  449.5   sim/theory=1.346
   ell=  564.5   sim/theory=1.484
   ell=  712.0   sim/theory=1.560
   ell=  897.0   sim/theory=1.608
   ell= 1157.0   sim/theory=1.640
   ell= 1522.0   sim/theory=1.631
   ell= 2004.5   sim/theory=1.603
   ell= 2640.0   sim/theory=1.640


## 1. It is **not** a measurement artifact

The measured galaxy auto is stored as `shot_noise_subtracted_signal` — the weighted Poisson shot noise (computed from the **sim's own painted $\bar n$**, $1/\bar n = 1.20\times10^{-6}\,$sr) is already removed. The HEALPix pixel window is carried by both sim and data and is added to the theory side; at nside2048 it is negligible until the last band. The sim galaxy field is a pixelized $\delta_g$ map on the binary 2400 deg² cap, self-normalized.

Below: subtracting a constant (shot-noise-like) term does **not** flatten the ratio, and multiplying by $W_{\rm pix}^2$ barely moves it — so neither residual shot noise nor pixwin explains the rising trend.

In [2]:
import healpy as hp
W = hp.pixwin(2048, lmax=3200)
Well = np.array([W[int(round(l))] for l in ell])
# best-fit constant + amplitude (C_sim = A*C_theory + N) over the gg likelihood band 200<=ell<=2500
m = (ell>=200)&(ell<=2500)
A, N = np.polyfit(th_gg[m], sim_gg[m], 1)
print(f"linear fit over likelihood band:  C_sim = {A:.3f}*C_theory + ({N:+.3e})")
print(f"W_pix^2 at the 13 bands: {np.round(Well**2,3)}  (only 0.86 at the last band)\n")
print(f"{'ell':>7} {'sim/theory':>11} {'x Wpix^2':>9} {'(sim-N)/theory':>15}")
for i,l in enumerate(ell):
    print(f"{l:7.1f} {obs[i]:11.3f} {obs[i]*Well[i]**2:9.3f} {(sim_gg[i]-N)/th_gg[i]:15.3f}")


linear fit over likelihood band:  C_sim = 1.184*C_theory + (+4.563e-07)
W_pix^2 at the 13 bands: [1.    0.999 0.999 0.998 0.997 0.996 0.993 0.989 0.982 0.971 0.95  0.914
 0.856]  (only 0.86 at the last band)

    ell  sim/theory  x Wpix^2  (sim-N)/theory
  143.5       1.092     1.092           1.061
  179.5       1.114     1.113           1.069
  227.0       1.268     1.266           1.206
  287.0       1.195     1.193           1.113
  359.5       1.327     1.323           1.212
  449.5       1.346     1.340           1.189
  564.5       1.484     1.474           1.271
  712.0       1.560     1.543           1.276
  897.0       1.608     1.580           1.244
 1157.0       1.640     1.592           1.180
 1522.0       1.631     1.549           1.055
 2004.5       1.603     1.466           0.885
 2640.0       1.640     1.403           0.720


## 2. It is **not** the $n(z)$ projection

The catalog selects halos by true redshift in $[0.63,0.98]$; the theory projects with the DESI calibrated true-$z$ kernel. They share the same mean ($\langle z\rangle=0.791$). The catalog kernel is *slightly* narrower because the hard cut trims the calibration-$n(z)$ tails — but the **projection-relevant** quantity $\int(dN/dz)^2 dz$ (which sets the Limber amplitude) differs by only **1.4%**, and is $\ell$-independent. So $n(z)$ cannot produce a 50% scale-dependent excess.

In [3]:
with h5py.File(MAP,"r") as h5: gcat = np.asarray(h5["galaxies"][:])
v = gcat[:,5]>0.5; zsim = gcat[v,2]
with h5py.File(NZH5,"r") as h5:
    zmid = np.asarray(h5["nz/desi/z_mid"][:]); zedges = np.asarray(h5["nz/desi/z_edges"][:])
    dndz_th = np.asarray(h5["nz/desi/nz_dndz_by_pz"][2])
nsim,_ = np.histogram(zsim, bins=zedges); dndz_sim = nsim/np.diff(zedges)
nrm = lambda x: x/np.trapezoid(x, zmid)
ns, nt = nrm(dndz_sim), nrm(dndz_th)
I_s, I_t = np.trapezoid(ns**2, zmid), np.trapezoid(nt**2, zmid)
sig = lambda n: np.sqrt(np.trapezoid((zmid-np.trapezoid(zmid*nrm(n),zmid))**2*nrm(n),zmid))
print(f"sigma_z  sim={sig(dndz_sim):.4f}  theory={sig(dndz_th):.4f}")
print(f"int (dN/dz)^2 dz  sim={I_s:.4f}  theory={I_t:.4f}  ->  sim/theory = {I_s/I_t:.4f}  (1.4% projection effect)")
fig,ax=plt.subplots(figsize=(6,3.6))
ax.plot(zmid, nt, label="theory kernel (DESI calib true-$z$)", lw=2)
ax.plot(zmid, ns, label="painted Abacus catalog", lw=2, ls="--")
ax.set_xlim(0.6,1.0); ax.set_xlabel("z"); ax.set_ylabel("$n(z)$ (unit norm)")
ax.set_title(f"pz3 $n(z)$: $\\int(dN/dz)^2$ ratio = {I_s/I_t:.3f}"); ax.legend(); fig.tight_layout()
fig.savefig(PLOTS/"diag_nz_comparison.pdf"); plt.show()


sigma_z  sim=0.0655  theory=0.0748
int (dN/dz)^2 dz  sim=4.2390  theory=4.1825  ->  sim/theory = 1.0135  (1.4% projection effect)


## 3. Build the bestfit theory and split $C_\ell^{gg}$ into 1-halo + 2-halo

We rebuild exactly the model the paste pipeline's `build_theory` uses (same merged bestfit params, `gg_transition_model="poweradd"`), then extract:
- the predicted comoving galaxy density $\bar n_{\rm gal}(z)$ and the implied surface density (should reproduce the **target**, since the HOD threshold was solved against the Tinker HMF);
- the 1-halo and 2-halo $C_\ell^{gg}$, obtained by re-running the Limber projection with `Pgg_tot_mat` swapped for its `Pgg_1h_kz_mat` / `Pgg_2h_kz_mat` components.

*(This cell runs the JAX model on CPU; ~1-3 min.)*

In [4]:
sys.path.insert(0, f"{REPO}/notebooks/xDESI/survey_measure")
sys.path.insert(0, f"{REPO}/notebooks/xDESI/abacus_paste")
import godmax_multiprobe_theory_utils as gmt
import stage31_pz1_backlight_validation as drv

config = drv.read_config(CONFIG)
cfg = drv.merge_bestfit_params(config); cfg["metadata"]["lmax"] = 3000
cfg = gmt.compute_desi_nbar_comoving(cfg)
pz_bin = drv.pz_bin_from_config(config)
pz_cfg = gmt.config_for_single_desi_pz(cfg, pz_bin)
gmt.ensure_godmax_import_paths(Path(pz_cfg.get("repo_root") or cfg.get("repo_root") or REPO))
from base_class import base_class
from get_Pkzs import get_Pkz
from get_radial_profiles import Profiles
from get_Cls import get_Cl
sp,hp_,an,op = gmt._params_for_model(pz_cfg, is_cmb_lensing=False); an["gg_transition_model"]="poweradd"
base = base_class(sp,hp_,an,op); prof = Profiles(sp,hp_,an,op,base_class_obj=base)
pkz  = get_Pkz(sp,hp_,an,op,Profiles_obj=prof); cls = get_Cl(sp,hp_,an,op,Pkz_obj=pkz)
A = lambda x: np.asarray(x)

zc=A(base.z_array); chi=A(base.chi_array); dchidz=A(base.dchi_dz_array); nbar_th=A(base.nbar_gal_comoving_array)
sr=(np.pi/180.0)**2
Nth = np.trapezoid(nbar_th*chi**2*dchidz, zc)*sr
ell_th = A(cls.ell_array); Cl_tot = A(cls.Cl_gal_gal_tot_mat[:,0,0])
pkz.Pgg_tot_mat = pkz.Pgg_1h_kz_mat; Cl_1h = A(get_Cl(sp,hp_,an,op,Pkz_obj=pkz).Cl_gal_gal_tot_mat[:,0,0])
pkz.Pgg_tot_mat = pkz.Pgg_2h_kz_mat; Cl_2h = A(get_Cl(sp,hp_,an,op,Pkz_obj=pkz).Cl_gal_gal_tot_mat[:,0,0])
f1 = np.interp(ell, ell_th, Cl_1h/Cl_tot)     # 1-halo fraction at the 13 band centers
print(f"THEORY galaxy surface density = {Nth:.1f}/deg2   (config target 334.86 ; painted 253.8)")
print(f"1-halo fraction f1 at bands: {np.round(f1,3)}")


THEORY galaxy surface density = 343.9/deg2   (config target 334.86 ; painted 253.8)
1-halo fraction f1 at bands: [0.061 0.084 0.114 0.151 0.208 0.283 0.377 0.49  0.604 0.719 0.818 0.888
 0.934]


### 3a. The $\bar n$ deficit — painted Abacus density vs theory

Computing the painted galaxies' **comoving** density on the model's own grid (using the model's $\chi(z)$, $d\chi/dz$ so no extra cosmology assumptions enter): the painted density is **0.743×** the theory density, **consistently across every populated redshift shell**. This is the HMF/abundance deficit (Tinker vs Abacus) projected through the HOD.

In [5]:
ra=np.radians(gcat[v,0]); dec=np.radians(gcat[v,1]); zg=gcat[v,2]
ra0,dec0,rad = np.radians(37.96875), np.radians(-34.953865257188454), np.radians(27.91480167872345)
incap = (np.sin(dec)*np.sin(dec0)+np.cos(dec)*np.cos(dec0)*np.cos(ra-ra0)) >= np.cos(rad)
Omega = 2400.036403392469*sr
edges = np.concatenate([[zc[0]-(zc[1]-zc[0])/2], 0.5*(zc[1:]+zc[:-1]), [zc[-1]+(zc[-1]-zc[-2])/2]])
cnt,_ = np.histogram(zg[incap], bins=edges); dz=np.diff(edges)
nbar_sim = np.where((chi>0)&(dz>0), cnt/(Omega*chi**2*dchidz*dz), 0.0)
sel = nbar_th>1e-7
w = np.where(sel, nbar_th*chi**2*dchidz, 0.0)
deficit = np.trapezoid(nbar_sim*w, zc)/np.trapezoid(nbar_th*w, zc)
boost = 1.0/deficit**2
print(f"painted cap galaxies = {incap.sum()}  ({incap.sum()/2400.036:.1f}/deg2)")
print(f"n(z)-weighted painted/theory comoving density = {deficit:.3f}")
print(f"=> 1-halo boost (1/nbar^2)  = {boost:.3f}")
fig,ax=plt.subplots(figsize=(6,3.8))
ax.plot(zc[sel], nbar_th[sel]*1e4, "o-", label="theory $\\bar n_{gal}(z)$ (Tinker HMF + HOD)")
ax.plot(zc[sel], nbar_sim[sel]*1e4, "s--", label="painted Abacus $\\bar n_{gal}(z)$")
ax.set_xlim(0.6,1.0); ax.set_xlabel("z"); ax.set_ylabel(r"$\bar n_{gal}\;[10^{-4}\,(h/{\rm Mpc})^3]$")
ax.set_title(f"Painted/theory = {deficit:.3f}  →  $1/\\bar n^2$ boost = {boost:.2f}"); ax.legend(); fig.tight_layout()
fig.savefig(PLOTS/"diag_nbar_deficit.pdf"); plt.show()


painted cap galaxies = 609170  (253.8/deg2)
n(z)-weighted painted/theory comoving density = 0.743
=> 1-halo boost (1/nbar^2)  = 1.813


### 3b. The decomposition fit — $1/\bar n^2$ explains the rising ratio

Fit the band-by-band ratio to a two-amplitude halo-model: $\;\text{sim}/\text{theory}(\ell)=a_{2h}\,(1-f_1)+a_{1h}\,f_1$, where $f_1(\ell)$ is the theory 1-halo fraction. The fitted **1-halo amplitude $a_{1h}$ matches the independent $1/\bar n^2$ prediction**, and $a_{2h}$ gives the residual bias offset.

In [6]:
Amat = np.vstack([1-f1, f1]).T
(a2, a1h), *_ = np.linalg.lstsq(Amat, obs, rcond=None)
print(f"a_2h (2-halo amplitude ratio) = {a2:.3f}   ->  b_sim/b_th = {np.sqrt(max(a2,0)):.3f}  (+{100*(np.sqrt(max(a2,0))-1):.1f}%)")
print(f"a_1h (1-halo amplitude ratio) = {a1h:.3f}")
print(f"independent 1/nbar^2 prediction = {boost:.3f}   (agreement: {100*a1h/boost:.0f}%)")

fit  = a2*(1-f1) + a1h*f1
mech = a2*(1-f1) + boost*f1
fig,axs=plt.subplots(1,2,figsize=(11,4))
ax=axs[0]
ax.plot(ell, obs, "ko", ms=7, label="measured sim/theory")
ax.plot(ell, fit, "-", color="C3", lw=2, label=f"fit: $a_{{2h}}$={a2:.2f}, $a_{{1h}}$={a1h:.2f}")
ax.plot(ell, mech, "--", color="C0", lw=2, label=f"$1/\\bar n^2$ mechanism ($a_{{1h}}$={boost:.2f})")
ax.plot(ell, a2*(1-f1)+1*f1, ":", color="0.5", label="no 1-halo boost ($a_{1h}$=1)")
ax.axhline(1,color="0.7",lw=0.8); ax.set_xscale("log"); ax.set_xlabel(r"$\ell$"); ax.set_ylabel(r"$C_\ell^{gg}$ sim / theory")
ax.set_title("Ratio is the 1-halo fraction scaled by the boost"); ax.legend(fontsize=8)
ax=axs[1]
fac = ell*(ell+1)/(2*np.pi)
ax.plot(ell, fac*sim_gg, "ko", ms=6, label="sim (pasted Abacus)")
ax.plot(ell, fac*th_gg, "-", color="0.4", lw=2, label="theory (as-is)")
ax.plot(ell, fac*th_gg*mech, "--", color="C0", lw=2, label=r"theory corrected ($\bar n\to\bar n_{\rm sim}$, $b\!\times\!1.076$)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel(r"$\ell$"); ax.set_ylabel(r"$D_\ell^{gg}=\ell(\ell+1)C_\ell/2\pi$")
ax.set_title("Corrected theory reproduces the sim"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(PLOTS/"diag_gg_decomposition.pdf"); plt.show()


a_2h (2-halo amplitude ratio) = 1.158   ->  b_sim/b_th = 1.076  (+7.6%)
a_1h (1-halo amplitude ratio) = 1.761
independent 1/nbar^2 prediction = 1.813   (agreement: 97%)


## 4. The physical mechanism

The halo-model galaxy auto power is
$$P_{gg}(k)=\underbrace{\frac{1}{\bar n^2}\!\int\! dM\,\frac{dn}{dM}\Big[2N_{\rm cen}N_{\rm sat}\,u(k|M)+N_{\rm sat}^2\,u(k|M)^2\Big]}_{\text{1-halo}}\;+\;\underbrace{b_g^2\,P_{\rm lin}(k)}_{\text{2-halo}},\qquad \bar n=\int dM\,\frac{dn}{dM}\,(N_{\rm cen}+N_{\rm sat}).$$

- The **2-halo** term is $\propto b_g^2$. Abacus halos at fixed mass are ~7.6% more biased than the analytic Tinker-2010 fit, giving $a_{2h}=b_{\rm sim}^2/b_{\rm th}^2\approx1.16$. This is $\ell$-flat and is the *expected* "HMF/halo" difference.

- The **1-halo** term is $\propto 1/\bar n^2$. The HOD threshold was solved so the **Tinker-HMF** integral gives $\bar n_{\rm th}\approx344/\mathrm{deg}^2$ (= the DESI target). Applied to the **actual Abacus halos**, the same HOD paints $\bar n_{\rm sim}=253.8/\mathrm{deg}^2$. The measured $\delta_g$ self-normalizes by $\bar n_{\rm sim}$, but the theory divides by $\bar n_{\rm th}^2$. The intra-halo satellite **pairs** (formed in the massive halos that host satellites) are realized at nearly the Tinker level, while the **mean density** $\bar n$ — dominated by the much more numerous low-mass centrals where the Abacus deficit lives — is 24% low. Dividing an (almost) unchanged pair count by a 24%-smaller $\bar n^2$ boosts the 1-halo by $(1/0.743)^2=1.81$.

Because $f_1(\ell)$ climbs from 6% at $\ell=144$ to 93% at $\ell=2640$, this constant 1-halo boost appears as a **ratio that rises with $\ell$** — precisely the observed shape. That the 1-halo *shape* is reproduced by a single amplitude (the fit works) is independent proof that the **satellite profile $u(k|M)$ is correct** and only its **normalization** is wrong.

### Why it looks like "50-60%" not "10-15%"
The genuine halo/HMF difference is two things: a ~7.6% bias offset (2-halo) and a ~24% number-density deficit (which sets $\bar n$). The bias offset is small and $\ell$-flat. But the number-density deficit enters the 1-halo term **quadratically** ($1/\bar n^2$), turning 24% into ~80%, and the 1-halo dominates exactly where the excess is largest (high $\ell$). The "$50\text{-}60\%$" is therefore the small HMF difference *quadratically amplified*, not a new ~50% discrepancy.

## 5. Corrections

The painting is internally correct (right profile, right shot-noise convention). The inconsistency is that **the theory assumes a galaxy density the simulation does not realize.** Two equally valid fixes depending on intent:

### (A) To make the sim↔theory comparison apples-to-apples (recommended for closure tests)
Recompute the theory with the **realized** density and the **Abacus** halo statistics, not the analytic target:
1. **Set the theory 1-halo normalization to the painted $\bar n_{\rm sim}(z)$** (the comoving density measured directly from the catalog, computed in §3a). Equivalently, multiply the theory 1-halo by $(\bar n_{\rm th}/\bar n_{\rm sim})^2\approx1.81$. This removes the dominant rising excess.
2. **Use the Abacus HMF and $b(M)$** (measured from the halo catalog) in the theory instead of Tinker-08/Tinker-10. This simultaneously fixes the $\bar n$ deficit *and* the +7.6% 2-halo bias — leaving sim and theory consistent to the percent level.

The orange dashed curve in §3b is this corrected theory (1-halo $\times1.81$, bias $\times1.076$): it reproduces the sim across all scales.

### (B) To make the *paste* reproduce the *target* (recommended if the paste must match the survey density)
**Re-solve the HOD mass threshold $M_*^{\rm thresh}(z)$ against the actual Abacus halo abundance** (not the Tinker HMF), so the painted catalog hits $\bar n=335/\mathrm{deg}^2$. Then $\bar n_{\rm sim}=\bar n_{\rm th}$, the 1-halo normalization matches by construction, and only the genuine ~7.6% Abacus-vs-Tinker bias remains (which is the honest "~10-15% from the HMF").

### Recommended verification (next step)
Measure the Abacus halo mass function $dn/d\ln M$ directly from the cap halo catalog and overlay the Tinker-08/10 HMF the theory uses, over $10^{12.5}$–$10^{14}\,M_\odot/h$ (the HOD-weighted range). This will (i) confirm the ~24% abundance deficit and reveal whether it is a genuine HMF difference at this cosmology or a **halo mass-definition / completeness offset** near the $10^{11.15}$ floor, and (ii) provide the Abacus $b(M)$ for fix (A2). The $\bar n_{\rm gal}(z)$ comparison in §3a already shows the deficit projected through the HOD; the direct HMF closes the loop.

### Out of scope (separate, already-tracked items)
The kSZ velocity-normalization residual and the catalog-$\pi$ pixel-window over-windowing are unrelated to $C_\ell^{gg}$ and are tracked separately.

## 6. The fix — hit the target survey density (and where the deficit comes from)

The painted density is **253.8/deg²**, only **0.758×** the 334.86/deg² target. Loading the full 52M-halo catalog and evaluating the HOD on it shows the 24% deficit factorizes into **two independent ~13% pieces**:

1. **HOD occupation interpolation suppression (paste-side, fixable).** `get_sim_maps.py:1211` evaluates `⟨N_cen⟩ = exp(logNcen_interp(z, lnM))` with a `interpax.Interpolator2D(method='monotonic')` on **log(N_cen)** over a coarse **24-point** mass grid (0.2 dex). Monotone-log interpolation through the sharp erf central turn-on **undershoots ⟨N_cen⟩ by ~16%**. (Centrals are drawn correctly — `bernoulli(key, ncen_mean)`, line 1229; the paste's log-interp eval reproduces the realized count to <1%, so there is *no* realization bug.) The theory does not suffer this — it trapezoid-integrates `Ncen_mat` directly.

2. **Genuine HMF/mass deficit (Abacus vs Tinker).** Even with accurate (linear/fine-grid) occupation, the HOD on the *actual* Abacus halos gives **290.9/deg²** vs the Tinker-tuned target 334.9 — Abacus c9999 has ~13% fewer halos in the occupied mass range, or the count-based `M200c` (`mass_provenance: Interpolated_N × ParticleMassHMsun`) is ~11% low relative to the SO mass the theory's c(M)/HMF assume.

`0.867 (interp) × 0.869 (HMF) = 0.753 ≈ 253.8/334.9`. The cell below reproduces this and computes the calibration knob.

In [7]:
import interpax, jax.numpy as jnp
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import brentq
Ncen_g=np.asarray(prof.Ncen_mat); Nsat_g=np.asarray(prof.Nsat_mat)
M_g=np.asarray(prof.M_array); z_g=np.asarray(base.z_array); lMn=np.log(M_g).astype('float32'); l10M=np.log10(M_g)
# paste's EXACT occupation eval (monotonic log-interp) and an accurate linear interp
iNc=interpax.Interpolator2D(jnp.asarray(z_g.astype('float32')),jnp.asarray(lMn),jnp.asarray(np.log(Ncen_g+1e-20).astype('float32')),method='monotonic',extrap=[-20,-20])
iNs=interpax.Interpolator2D(jnp.asarray(z_g.astype('float32')),jnp.asarray(lMn),jnp.asarray(np.log(Nsat_g+1e-20).astype('float32')),method='monotonic',extrap=[-20,-20])
iNc_lin=RegularGridInterpolator((z_g,l10M),Ncen_g,bounds_error=False,fill_value=None)
iNs_lin=RegularGridInterpolator((z_g,l10M),Nsat_g,bounds_error=False,fill_value=None)
HALO=f"{REPO}/data/xDESI/processed/abacus_backlight/stage31_pz3_cap2400_lmax3000_gk1000_60param_warm100_checkpoint_000550/halos/abacus_c9999_ph9999_pz3cap2400_hmcbestfit_z0p63_0p98_logMgt11p147538_halos.h5"
ze=np.linspace(0.5,1.1,121); me=np.linspace(11.0,15.6,231); Hh=np.zeros((ze.size-1,me.size-1))
with h5py.File(HALO,"r") as h5:           # chunked cap histogram (memory-safe)
    n=h5["z"].shape[0]
    for s in range(0,n,4_000_000):
        e=min(s+4_000_000,n)
        rr=np.radians(h5["ra_deg"][s:e].astype("f8")); dd=np.radians(h5["dec_deg"][s:e].astype("f8"))
        mm=(np.sin(dd)*np.sin(dec0)+np.cos(dd)*np.cos(dec0)*np.cos(rr-ra0))>=np.cos(rad)
        Hh+=np.histogram2d(h5["z"][s:e][mm].astype("f8"),h5["log10M200c_hMsun"][s:e][mm].astype("f8"),bins=[ze,me])[0]
area=2400.036403392469
zct=0.5*(ze[1:]+ze[:-1]); m10=0.5*(me[1:]+me[:-1]); ZZ,MM=np.meshgrid(zct,m10,indexing="ij")
shp=ZZ.shape
def dens(shift, kind):   # shift in dex on halo log10 mass; kind 'paste'(log-mono) or 'lin'
    z=np.clip(ZZ,z_g.min(),z_g.max()); m=np.clip(MM+shift,l10M.min(),l10M.max())
    if kind=="paste":
        zf=jnp.asarray(z.ravel().astype('f4')); lf=jnp.asarray((m.ravel()*np.log(10)).astype('f4'))
        nc=np.clip(np.nan_to_num(np.asarray(jnp.exp(iNc(zf,lf)))),0,1).reshape(shp)
        ns=np.clip(np.nan_to_num(np.asarray(jnp.exp(iNs(zf,lf)))),0,None).reshape(shp)
    else:
        pts=np.stack([z,m],axis=-1); nc=np.clip(iNc_lin(pts),0,1); ns=np.clip(iNs_lin(pts),0,None)
    return float(np.sum(Hh*(nc+ns)))/area
tgt=334.86400893955306
d_paste, d_lin = dens(0.0,"paste"), dens(0.0,"lin")
print(f"target (Tinker theory)                 : {tgt:6.1f}/deg2")
print(f"HOD on Abacus, accurate (linear) interp: {d_lin:6.1f}/deg2   (ratio {d_lin/tgt:.3f}  <- HMF/mass deficit)")
print(f"HOD on Abacus, paste log-monotonic eval : {d_paste:6.1f}/deg2   (ratio {d_paste/d_lin:.3f}  <- interp suppression)")
print(f"realized painted (measured)            : {253.8:6.1f}/deg2   (matches paste eval to <1%)")
sh_lin =brentq(lambda s: dens(s,"lin")  -tgt, 0.0, 0.8, xtol=1e-4)
sh_past=brentq(lambda s: dens(s,"paste")-tgt, 0.0, 0.8, xtol=1e-4)
print(f"\nFIX-2 knob if interp ALSO fixed (linear): dlog10M = {sh_lin:.3f} dex  (M200c x{10**sh_lin:.3f}, +{100*(10**sh_lin-1):.0f}%)  -> {dens(sh_lin,'lin'):.1f}/deg2")
print(f"single knob absorbing BOTH (keep paste) : dlog10M = {sh_past:.3f} dex  (M200c x{10**sh_past:.3f}, +{100*(10**sh_past-1):.0f}%)  -> {dens(sh_past,'paste'):.1f}/deg2")
fig,ax=plt.subplots(figsize=(6.4,3.8))
labels=["paste\n(realized)","+ fix interp\n(linear)","+ mass knob","target\n(survey)"]
vals=[d_paste, d_lin, dens(sh_lin,"lin"), tgt]
ax.bar(labels, vals, color=["#c0392b","#e67e22","#27ae60","0.4"])
for i,vv in enumerate(vals): ax.text(i, vv+4, f"{vv:.0f}", ha="center", fontsize=9)
ax.axhline(tgt, color="0.4", ls="--", lw=1); ax.set_ylabel(r"$\bar n_{\rm gal}$ [/deg$^2$]")
ax.set_title("Two-step fix to the target survey density"); fig.tight_layout()
fig.savefig(PLOTS/"diag_density_fix.pdf"); plt.show()


target (Tinker theory)                 :  334.9/deg2
HOD on Abacus, accurate (linear) interp:  290.9/deg2   (ratio 0.869  <- HMF/mass deficit)
HOD on Abacus, paste log-monotonic eval :  252.2/deg2   (ratio 0.867  <- interp suppression)
realized painted (measured)            :  253.8/deg2   (matches paste eval to <1%)

FIX-2 knob if interp ALSO fixed (linear): dlog10M = 0.046 dex  (M200c x1.112, +11%)  -> 334.9/deg2
single knob absorbing BOTH (keep paste) : dlog10M = 0.092 dex  (M200c x1.235, +23%)  -> 334.9/deg2


### Recommended fix (easiest, physically consistent, no profile re-paint)

The galaxy population is **cheap to re-run** (HOD draws on the existing halo catalog), and the κ/y/τ profile maps are unaffected, so only the galaxy catalog + `C_gg` need regenerating.

**Step 1 — remove the interpolation suppression (recovers ~13%, also reduces the bias offset).**
In `get_sim_maps.py` `get_hod_params`/`_setup_galmap`, evaluate ⟨N_cen⟩ accurately through the turn-on:
- bump the HOD occupation **mass-grid resolution** (`nM` 24 → ~128), and/or
- interpolate **N_cen linearly** (or interpolate the erf argument) instead of `monotonic` on `log(N_cen)`.
This makes the paint reproduce the *same* ⟨N_cen⟩ the theory integrates; the recovered galaxies are low-mass low-bias centrals, so it also pulls the +7.6% 2-halo bias toward theory.

**Step 2 — close the residual ~13% HMF/mass deficit with one physical knob.**
Add a single `halo_mass_rescale` (≈ **+11%**, `Δlog10M ≈ 0.046 dex`) — or an HOD turn-on shift — applied to the Abacus `M200c` inside the galaxy-population step, tuned (3–4 cheap galaxy-only re-draws) so the **realized** density = 334.86/deg². Interpretation: a count-based `M200c` → SO-`M200c` mass-definition correction. If you skip Step 1, a single knob of `Δlog10M ≈ 0.09 dex` absorbs *both* effects at once (tune against the realized density).

**Why this makes sim/theory/data comparable.** Once `n̄_sim = n̄_target`, the measured `δ_g = n/n̄_sim − 1` and the theory 1-halo term divide by the *same* `n̄`, so the `(n̄_th/n̄_sim)² = 1.8×` 1-halo boost **disappears** and `C_ℓ^{gg}` agrees with theory up to the genuine few-% Abacus bias — and the catalog is a faithful survey-density mock for data comparison.

**Zero-re-paint alternative (comparison only).** Feed the realized `n̄_sim(z)` (§3a) into the theory's 1-halo normalization (×1.81) — the orange "corrected theory" curve in §3b — no paste changes.

**Verify after the fix:** re-draw galaxies, re-measure `C_gg`, confirm sim/theory flattens to ≈ 1 (×bias²) across ℓ and realized `n̄ = 334.86/deg²`.